In [ ]:
# Ô CODE 1 — Battle of 3 Variants: chọn duy nhất một model cho mỗi tài khoản Kaggle
import gc
import json
import math
import random
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

# MỖI TÀI KHOẢN CHỈ CHỌN MỘT TRONG BA GIÁ TRỊ SAU:
# "ENHANCED", "UNCERTAINTY_AWARE", "EPSILON_GREEDY"
MODEL_TO_RUN = "UNCERTAINTY_AWARE"
RUN_TRAINING = True
RESUME = True
SEEDS = tuple(range(44, 64))  # Đúng 20 seed độc lập: 44..63.
SEED = 43  # Chỉ dùng để fit Frozen Scaler chung, không phải seed huấn luyện.
CURRENT_RUN_SEED = SEEDS[0]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA = 0.95
NN_LR = 5e-5
VAE_LR = 1e-3
COST_LR = 1e-3
EPISODES = 45
SARSA_ALPHA = 0.60
Q_WEIGHT_DECAY = 1e-4
VAE_BETA_KL = 0.01
LATENT_DIM = 16
BATCH_SIZE = 128
VAE_BATCH_SIZE = 256
BOOTSTRAP_TRAJECTORIES = 5
BOOTSTRAP_UPDATES = 100
ONLINE_AUX_UPDATES = 1
REPLAY_CAPACITY = 50_000
BALANCE_INIT = 1_000.0
TRANSACTION_FEE = 0.001
W_RISK = 0.15
W_STABILITY = 0.05
ZETA = 0.05
BETA_DECAY = 0.91
BETA_MIN = 0.01
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT = 2.0

# Cấu hình khóa cứng sau screening.
# test12345.ipynb chỉ khai báo backbone chung; ba tham số epsilon dùng mặc định UCB_2.py.
LOCKED_CONFIGS = {
    "ENHANCED": {
        "label": "Enhanced UCB-VAE", "beta_0": 0.03, "use_cost": False,
        "robust_loss": True, "weight_decay": Q_WEIGHT_DECAY,
        "reward_shaping": True, "kl_reduction": "sum", "vae_beta_kl": VAE_BETA_KL,
    },
    "UNCERTAINTY_AWARE": {
        "label": "Uncertainty-Aware UCB-VAE", "beta_0": 0.03, "use_cost": True,
        "robust_loss": True, "weight_decay": Q_WEIGHT_DECAY,
        "reward_shaping": True, "kl_reduction": "sum", "vae_beta_kl": VAE_BETA_KL,
    },
    "EPSILON_GREEDY": {
        "label": "Epsilon-Greedy", "use_cost": False, "robust_loss": True,
        "weight_decay": 0.0, "reward_shaping": True,
        "epsilon_init": 1.0, "epsilon_decay": 0.95, "epsilon_min": 0.05,
    },
}
if MODEL_TO_RUN not in LOCKED_CONFIGS:
    raise ValueError(f"MODEL_TO_RUN phải thuộc {list(LOCKED_CONFIGS)}, nhận được {MODEL_TO_RUN!r}")
SELECTED_CONFIG = dict(LOCKED_CONFIGS[MODEL_TO_RUN])
MODEL_TAG = MODEL_TO_RUN.lower()


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print({
    "model": MODEL_TO_RUN, "config": SELECTED_CONFIG, "seeds": SEEDS,
    "device": str(DEVICE), "runs_on_this_account": len(SEEDS),
})


In [ ]:
# Ô CODE 2 — Dữ liệu HPG BAD, môi trường và Frozen Standardization
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]
STATE_NAMES = ["Price", "Balance", "Position", "MACD", "RSI", "CCI", "ADX"]


def load_hpg_bad() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing:
            raise ValueError(f"HPG BAD {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        numeric = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if numeric.isna().any().any():
            raise ValueError(f"HPG BAD {label} chứa NaN/giá trị không hợp lệ.")
        frame[REQUIRED_COLUMNS[1:]] = numeric
    if train["time"].max() >= test["time"].min():
        raise ValueError("Train/Test chồng lấn thời gian; dừng để tránh leakage.")
    return train.reset_index(drop=True), test.reset_index(drop=True)


def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray(
        [row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]],
        dtype=np.float32,
    )


class TradingEnv:
    """Single-asset, long-only; action là số cổ phiếu giao dịch trong [-5, 5]."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2:
            raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True)
        self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash)
        self.reset()

    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash
        self.portfolio_history = [self.initial_cash]
        self.var_targets: List[float] = []
        return self._state()

    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)

    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            affordable = int(self.cash // (price * (1.0 + TRANSACTION_FEE)))
            executed = min(int(requested), affordable)
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed

    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping:
            reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value))
        self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {
            "raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return),
            "var_target": float(var_target), "drawdown": float(drawdown),
            "executed_action": int(executed), "portfolio_value": float(portfolio_value),
        }
        return self._state(), float(reward), done, info


@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray

    def transform(self, states: np.ndarray) -> np.ndarray:
        x = np.asarray(states, dtype=np.float32)
        return (x - self.mean) / self.std


def train_only_calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    """Ước lượng đủ 7 chiều state bằng random rollout chỉ trên tập train."""
    rng = np.random.default_rng(SEED)
    states: List[np.ndarray] = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False)
        state, done = env.reset(), False
        while not done:
            states.append(state.copy())
            action = int(rng.choice(ACTION_VALUES))
            state, _, done, _ = env.step(action)
    return np.asarray(states, dtype=np.float32)


train_hpg, test_hpg = load_hpg_bad()
calibration = train_only_calibration_states(train_hpg)
frozen_mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenScaler(frozen_mean.copy(), frozen_std.copy())
FROZEN_SCALER.mean.setflags(write=False)
FROZEN_SCALER.std.setflags(write=False)
print(pd.DataFrame({"feature": STATE_NAMES, "train_mean": frozen_mean, "train_std": frozen_std}).to_string(index=False))
print({"train": len(train_hpg), "test": len(test_hpg), "train_end": train_hpg.time.max(), "test_start": test_hpg.time.min()})


In [ ]:
# Ô CODE 3 — Q-Network, EpistemicVAE và Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class EpistemicVAE(nn.Module):
    """Joint VAE: 7 state + 11 action one-hot → latent 16 → reconstruction 18."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        reconstructed = self.decoder(mu + torch.randn_like(std) * std)
        return reconstructed, mu, logvar

    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        """Novelty với KLD sum/mean tùy ablation và probe z⁺=μ+1.96σ."""
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl_terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum":
            kl = torch.sum(kl_terms, dim=-1)
        elif kl_reduction == "mean":
            kl = torch.mean(kl_terms, dim=-1)
        else:
            raise ValueError("kl_reduction phải là 'sum' hoặc 'mean'.")
        std = torch.exp(0.5 * logvar)
        reconstructed_95 = self.decoder(mu + 1.96 * std)
        reconstruction_error = torch.norm(x - reconstructed_95, p=2, dim=-1)
        return kl + reconstruction_error


class CostNetwork(nn.Module):
    """Ước lượng VaR không âm từ concat(s_scaled, action_onehot) ∈ R^18."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)


def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE(), "\n", CostNetwork())


In [ ]:
# Ô CODE 4 — Loss, replay và cơ chế Confidence-Weighted Risk Penalty
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)


def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl


def cost_loss(predicted_var, var_target):
    return F.huber_loss(predicted_var, var_target)


class ReplayBuffer:
    def __init__(self, capacity: int = REPLAY_CAPACITY):
        self.capacity = int(capacity)
        self.states: List[np.ndarray] = []
        self.actions: List[int] = []
        self.var_targets: List[float] = []

    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32))
            self.actions.append(int(action))
            self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]
            del self.actions[:overflow]
            del self.var_targets[:overflow]

    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0:
            raise RuntimeError("Không thể sample replay buffer rỗng.")
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return (
            np.asarray([self.states[i] for i in indices]),
            np.asarray([self.actions[i] for i in indices]),
            np.asarray([self.var_targets[i] for i in indices], dtype=np.float32),
        )

    def __len__(self):
        return len(self.states)


def action_scores(q_network, vae, cost_network, state, beta, config, collect_details=False):
    """Q - confidence·VaR + beta·novelty_scaled theo đúng đặc tả."""
    scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1)
    action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval()
    if cost_network is not None:
        cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        calibrated_penalty = torch.zeros_like(q_values)
        predicted_var = torch.zeros_like(q_values)
        if config["use_cost"]:
            predicted_var = cost_network(state_batch, action_batch)
            confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
            calibrated_penalty = torch.where(
                predicted_var < ZETA,
                torch.zeros_like(predicted_var),
                confidence * predicted_var,
            )
        scores = q_values - calibrated_penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None
    if collect_details:
        details = {
            "novelty": novelty_raw.detach().cpu().numpy(),
            "predicted_var": predicted_var.detach().cpu().numpy(),
            "penalty": calibrated_penalty.detach().cpu().numpy(),
        }
    return int(ACTION_VALUES[action_index]), action_index, details


def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config):
    raw_states, action_indices, var_targets = replay.sample(VAE_BATCH_SIZE)
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE)
    actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions)
    target = torch.cat([states, actions], dim=-1)
    loss_vae = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True)
    loss_vae.backward()
    torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0)
    vae_optimizer.step()

    loss_cost_value = np.nan
    if config["use_cost"]:
        targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
        predicted = cost_network(states, actions)
        loss_c = cost_loss(predicted, targets)
        cost_optimizer.zero_grad(set_to_none=True)
        loss_c.backward()
        torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0)
        cost_optimizer.step()
        loss_cost_value = float(loss_c.detach().cpu())
    return float(loss_vae.detach().cpu()), loss_cost_value


def random_bootstrap(replay: ReplayBuffer):
    rng = np.random.default_rng(CURRENT_RUN_SEED)
    for _ in range(BOOTSTRAP_TRAJECTORIES):
        env = TradingEnv(train_hpg, reward_shaping=False)
        state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11))
            states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index]))
            var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)


def collect_episode(env, q_network, vae, cost_network, beta, config):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy())
        rewards.append(reward); actions.append(action_index); dones.append(done)
        var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets


def q_update(q_network, optimizer, trajectory, robust: bool):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
    an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE)
    losses: List[float] = []
    q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return losses


In [ ]:
# Ô CODE 5 — Huấn luyện duy nhất model đã chọn, checkpoint sau mỗi seed
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)


def epsilon_action(q_network: QNetwork, state: np.ndarray, epsilon: float, explore: bool) -> Tuple[int, int]:
    if explore and np.random.random() < float(epsilon):
        index = int(np.random.randint(0, len(ACTION_VALUES)))
    else:
        scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
        tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
        q_network.eval()
        with torch.no_grad(): index = int(torch.argmax(q_network(tensor).squeeze(0)).item())
    return int(ACTION_VALUES[index]), index


def collect_epsilon_episode(env, q_network, epsilon: float):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index = epsilon_action(q_network, state, epsilon, explore=True)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy())
        rewards.append(reward); actions.append(action_index); dones.append(done)
        var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets


def evaluate_locked(q_network, vae, cost_network, control_value, config, data, previous_row=None):
    env = TradingEnv(evaluation_frame(data, previous_row), reward_shaping=config["reward_shaping"])
    state, done = env.reset(), False
    while not done:
        if MODEL_TO_RUN == "EPSILON_GREEDY":
            action, _ = epsilon_action(q_network, state, epsilon=0.0, explore=False)
        else:
            action, _, _ = action_scores(q_network, vae, cost_network, state, control_value, config)
        state, _, done, _ = env.step(action)
    return np.asarray(env.portfolio_history, dtype=np.float64), np.asarray(env.var_targets, dtype=np.float64)


def period_metrics(portfolio: np.ndarray, dates: Sequence[pd.Timestamp], var_targets=None) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0])
    roi = float(profit / max(abs(portfolio[0]), 1e-8) * 100.0)
    dates = pd.Series(dates).reset_index(drop=True)
    days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = float(portfolio[-1] / max(portfolio[0], 1e-8))
    arr = float((ratio ** (365.25 / days) - 1.0) * 100.0) if ratio > 0 else -100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else float((annual_return - RISK_FREE_RATE_PERCENT) / volatility)
    peaks = np.maximum.accumulate(portfolio)
    mdd = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    violations = int(np.sum(np.asarray(var_targets) > ZETA)) if var_targets is not None else 0
    return {"profit": profit, "roi": roi, "arr": arr, "sharpe": sharpe, "max_drawdown": mdd, "violations": violations}


def run_locked_seed(seed: int):
    global CURRENT_RUN_SEED
    CURRENT_RUN_SEED = int(seed)
    set_seed(seed)
    config = dict(SELECTED_CONFIG)
    q_network = QNetwork().to(DEVICE)
    # Luôn khởi tạo theo cùng thứ tự RNG; epsilon không cập nhật hai mạng phụ trợ.
    vae, cost_network = EpistemicVAE().to(DEVICE), CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=VAE_LR) if MODEL_TO_RUN != "EPSILON_GREEDY" else None
    cost_optimizer = torch.optim.Adam(cost_network.parameters(), lr=COST_LR) if config.get("use_cost", False) else None
    replay = ReplayBuffer(); q_losses: List[float] = []; vae_losses: List[float] = []; cost_losses: List[float] = []
    curve_rows: List[Dict[str, float]] = []

    if MODEL_TO_RUN != "EPSILON_GREEDY":
        random_bootstrap(replay)
        for _ in range(BOOTSTRAP_UPDATES):
            lv, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
            vae_losses.append(lv)
            if np.isfinite(lc): cost_losses.append(lc)

    for episode in range(EPISODES):
        if MODEL_TO_RUN == "EPSILON_GREEDY":
            control = max(config["epsilon_min"], config["epsilon_init"] * config["epsilon_decay"] ** episode)
            env = TradingEnv(train_hpg, reward_shaping=config["reward_shaping"])
            trajectory = collect_epsilon_episode(env, q_network, control)
        else:
            control = max(BETA_MIN, config["beta_0"] * BETA_DECAY ** episode)
            env = TradingEnv(train_hpg, reward_shaping=config["reward_shaping"])
            trajectory = collect_episode(env, q_network, vae, cost_network, control, config)
            replay.add(trajectory[0], trajectory[3], trajectory[6])
        q_losses.extend(q_update(q_network, q_optimizer, trajectory, config["robust_loss"]))
        if MODEL_TO_RUN != "EPSILON_GREEDY":
            for _ in range(ONLINE_AUX_UPDATES):
                lv, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
                vae_losses.append(lv)
                if np.isfinite(lc): cost_losses.append(lc)

        train_portfolio, train_var = evaluate_locked(q_network, vae, cost_network, control, config, train_hpg)
        test_portfolio, test_var = evaluate_locked(q_network, vae, cost_network, control, config, test_hpg, train_hpg.iloc[-1])
        train_m = period_metrics(train_portfolio, train_hpg["time"], train_var)
        test_m = period_metrics(test_portfolio, test_hpg["time"], test_var)
        for split, values in (("train", train_m), ("test", test_m)):
            curve_rows.append({"model": config["label"], "seed": seed, "episode": episode + 1, "split": split, **values})

    final_portfolio, final_var = evaluate_locked(q_network, vae, cost_network, control, config, test_hpg, train_hpg.iloc[-1])
    final = period_metrics(final_portfolio, test_hpg["time"], final_var)
    metric_row = {
        "model_key": MODEL_TO_RUN, "model": config["label"], "seed": seed,
        "episodes": EPISODES, "beta_0": config.get("beta_0", np.nan),
        "epsilon_init": config.get("epsilon_init", np.nan),
        "epsilon_decay": config.get("epsilon_decay", np.nan),
        "epsilon_min": config.get("epsilon_min", np.nan), **final,
        "q_loss_mean": float(np.mean(q_losses)) if q_losses else np.nan,
        "vae_loss_mean": float(np.mean(vae_losses)) if vae_losses else np.nan,
        "cost_loss_mean": float(np.mean(cost_losses)) if cost_losses else np.nan,
    }
    loss_rows = []
    for loss_name, values in (("Q loss", q_losses), ("VAE loss", vae_losses), ("Cost loss", cost_losses)):
        for index, value in enumerate(values):
            loss_rows.append({"model": config["label"], "seed": seed, "loss_name": loss_name, "update": index, "loss": value})
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metric_row, curve_rows, loss_rows


METRICS_CSV = OUTPUT_DIR / f"battle_{MODEL_TAG}_metrics.csv"
CURVES_CSV = OUTPUT_DIR / f"battle_{MODEL_TAG}_curves.csv"
LOSSES_CSV = OUTPUT_DIR / f"battle_{MODEL_TAG}_losses.csv"
ERRORS_CSV = OUTPUT_DIR / f"battle_{MODEL_TAG}_errors.csv"


def append_frame(path: Path, frame: pd.DataFrame) -> None:
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


completed = set()
if RESUME and METRICS_CSV.exists():
    completed = set(pd.read_csv(METRICS_CSV)["seed"].astype(int).tolist())

if RUN_TRAINING:
    for position, seed in enumerate(SEEDS, start=1):
        if seed in completed:
            print(f"[{position}/20] skip seed={seed}: đã checkpoint")
            continue
        started = time.perf_counter()
        try:
            metric_row, curve_rows, loss_rows = run_locked_seed(seed)
            append_frame(CURVES_CSV, pd.DataFrame(curve_rows))
            if loss_rows: append_frame(LOSSES_CSV, pd.DataFrame(loss_rows))
            # Metrics ghi cuối cùng và đóng vai trò completion marker cho RESUME.
            append_frame(METRICS_CSV, pd.DataFrame([metric_row]))
            print(
                f"[{position}/20] {SELECTED_CONFIG['label']} seed={seed} | "
                f"profit={metric_row['profit']:.4f} | ROI={metric_row['roi']:.3f}% | "
                f"ARR={metric_row['arr']:.3f}% | Sharpe={metric_row['sharpe']:.4f} | "
                f"elapsed={time.perf_counter()-started:.1f}s"
            )
        except Exception as error:
            append_frame(ERRORS_CSV, pd.DataFrame([{
                "model": SELECTED_CONFIG["label"], "seed": seed,
                "error_type": type(error).__name__, "error": str(error),
                "traceback": traceback.format_exc(),
            }]))
            raise
else:
    print("RUN_TRAINING=False: bỏ qua training; có thể chạy Ô CODE 6 để gộp kết quả đã upload.")


In [ ]:
# Ô CODE 6 — Gộp kết quả 3 tài khoản và phân tích overfitting
def discover_result_files(pattern: str) -> List[Path]:
    files = list(OUTPUT_DIR.glob(pattern))
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists(): files.extend(kaggle_input.rglob(pattern))
    # Resolve/dedupe để không đọc cùng file hai lần.
    return list(dict.fromkeys(path.resolve() for path in files))


def load_combined(pattern: str, required: Sequence[str]) -> pd.DataFrame:
    frames = []
    for path in discover_result_files(pattern):
        try:
            frame = pd.read_csv(path)
            if set(required).issubset(frame.columns): frames.append(frame)
        except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
            print(f"Bỏ qua {path}: {error}")
    if not frames: return pd.DataFrame(columns=list(required))
    return pd.concat(frames, ignore_index=True).drop_duplicates()


all_metrics = load_combined("battle_*_metrics.csv", ["model", "seed", "profit", "roi", "arr", "sharpe", "max_drawdown", "violations"])
all_curves = load_combined("battle_*_curves.csv", ["model", "seed", "episode", "split", "profit", "roi", "arr", "sharpe"])
all_losses = load_combined("battle_*_losses.csv", ["model", "seed", "loss_name", "update", "loss"])
if all_metrics.empty:
    raise RuntimeError("Chưa tìm thấy battle_*_metrics.csv trong OUTPUT_DIR hoặc /kaggle/input.")
all_metrics.to_csv(OUTPUT_DIR / "battle_all_metrics_combined.csv", index=False)
if not all_curves.empty: all_curves.to_csv(OUTPUT_DIR / "battle_all_curves_combined.csv", index=False)
if not all_losses.empty: all_losses.to_csv(OUTPUT_DIR / "battle_all_losses_combined.csv", index=False)

summary = all_metrics.groupby("model").agg(
    seeds=("seed", "nunique"),
    profit_mean=("profit", "mean"), profit_std=("profit", "std"),
    roi_mean=("roi", "mean"), roi_std=("roi", "std"),
    arr_mean=("arr", "mean"), arr_std=("arr", "std"),
    sharpe_mean=("sharpe", "mean"), sharpe_std=("sharpe", "std"),
    max_drawdown_mean=("max_drawdown", "mean"),
    violations_mean=("violations", "mean"),
).reset_index()
SUMMARY_CSV = OUTPUT_DIR / "battle_three_variants_mean_std.csv"
SUMMARY_JSON = OUTPUT_DIR / "battle_three_variants_report.json"
summary.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps({
    "locked_configs": LOCKED_CONFIGS,
    "seed_range": list(SEEDS),
    "summary": json.loads(summary.to_json(orient="records")),
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("### Battle of 3 Variants — locked configs, Seeds 44–63")
try: print(summary.to_markdown(index=False, floatfmt=".6f"))
except ImportError: print(summary.to_string(index=False))
missing = sorted(set(spec["label"] for spec in LOCKED_CONFIGS.values()) - set(all_metrics["model"]))
if missing:
    print("Chưa có CSV của:", missing, "— upload output từ các tài khoản còn lại rồi chạy lại cell này.")

# Biểu đồ tổng kết các metric cuối kỳ trên 20 seed.
metrics_to_plot = [
    ("profit_mean", "profit_std", "Final Profit"), ("roi_mean", "roi_std", "ROI (%)"),
    ("arr_mean", "arr_std", "ARR (%)"), ("sharpe_mean", "sharpe_std", "Sharpe Ratio"),
    ("max_drawdown_mean", None, "Max Drawdown (%)"), ("violations_mean", None, "Constraint Violations"),
]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (mean_col, std_col, title) in zip(axes.ravel(), metrics_to_plot):
    errors = summary[std_col] if std_col else None
    ax.bar(summary["model"], summary[mean_col], yerr=errors, capsize=4)
    ax.set_title(title); ax.grid(axis="y", alpha=0.25); ax.tick_params(axis="x", rotation=18)
plt.tight_layout()
FINAL_METRICS_PNG = OUTPUT_DIR / "battle_final_metrics_mean_std.png"
fig.savefig(FINAL_METRICS_PNG, dpi=220, bbox_inches="tight")
plt.show()

# Generalization Gap theo episode: Profit, ROI, ARR và Sharpe train/test.
if not all_curves.empty:
    curve_mean = all_curves.groupby(["model", "episode", "split"])[["profit", "roi", "arr", "sharpe"]].mean().reset_index()
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for ax, metric, title in zip(axes.ravel(), ["profit", "roi", "arr", "sharpe"], ["Final Profit", "ROI (%)", "ARR (%)", "Sharpe Ratio"]):
        for (model, split), group in curve_mean.groupby(["model", "split"]):
            style = "-" if split == "train" else "--"
            ax.plot(group["episode"], group[metric], linestyle=style, label=f"{model} — {split}")
        ax.set(title=f"Generalization Gap — {title}", xlabel="Episode", ylabel=title)
        ax.grid(alpha=0.25); ax.legend(fontsize=7)
    plt.tight_layout()
    GENERALIZATION_PNG = OUTPUT_DIR / "battle_generalization_gap_profit_roi_arr_sharpe.png"
    fig.savefig(GENERALIZATION_PNG, dpi=220, bbox_inches="tight")
    plt.show()

# Phân phối ROI/ARR/Sharpe qua 20 seed — quan sát variance, outlier và độ bền.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, title in zip(axes, ["roi", "arr", "sharpe"], ["ROI (%)", "ARR (%)", "Sharpe Ratio"]):
    groups = [group[metric].dropna().to_numpy() for _, group in all_metrics.groupby("model")]
    labels = [name for name, _ in all_metrics.groupby("model")]
    ax.boxplot(groups, tick_labels=labels, showmeans=True)
    ax.set_title(f"20-seed distribution — {title}"); ax.grid(axis="y", alpha=0.25); ax.tick_params(axis="x", rotation=18)
plt.tight_layout()
DISTRIBUTION_PNG = OUTPUT_DIR / "battle_roi_arr_sharpe_distributions.png"
fig.savefig(DISTRIBUTION_PNG, dpi=220, bbox_inches="tight")
plt.show()

# Loss convergence trung bình; epsilon chỉ có Q loss, Enhanced không có Cost loss.
if not all_losses.empty:
    loss_mean = all_losses.groupby(["model", "loss_name", "update"])["loss"].mean().reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, loss_name in zip(axes, ["Q loss", "VAE loss", "Cost loss"]):
        subset = loss_mean[loss_mean["loss_name"] == loss_name]
        for model, group in subset.groupby("model"):
            ax.plot(group["update"], np.maximum(group["loss"], 1e-12), label=model)
        ax.set(title=loss_name, xlabel="Update", ylabel="Loss"); ax.set_yscale("log"); ax.grid(alpha=0.25)
        if not subset.empty: ax.legend(fontsize=8)
    plt.tight_layout()
    LOSSES_PNG = OUTPUT_DIR / "battle_loss_convergence.png"
    fig.savefig(LOSSES_PNG, dpi=220, bbox_inches="tight")
    plt.show()

print({
    "models_loaded": sorted(all_metrics["model"].unique()),
    "seeds_per_model": all_metrics.groupby("model")["seed"].nunique().to_dict(),
    "warning": "Chỉ kết luận sau khi mỗi model đủ đúng 20 seed 44–63.",
})
saved_outputs = [
    SUMMARY_CSV, SUMMARY_JSON, FINAL_METRICS_PNG, DISTRIBUTION_PNG,
    OUTPUT_DIR / "battle_all_metrics_combined.csv",
]
if not all_curves.empty: saved_outputs.extend([
    OUTPUT_DIR / "battle_all_curves_combined.csv", GENERALIZATION_PNG,
])
if not all_losses.empty: saved_outputs.extend([
    OUTPUT_DIR / "battle_all_losses_combined.csv", LOSSES_PNG,
])
print("\n=== FILE KẾT QUẢ ĐÃ LƯU ===")
for path in saved_outputs: print("-", path)
